# Lesson 3: working with state spaces

This lesson can be downloaded as a notebook, a notebook for colab and a python file [here](https://marmote.gitlabpages.inria.fr/marmote/python_downloads.html)

This C++ notebook follows the same sequence as the Python lesson and uses the `MarmoteSet` hierarchy directly.

**Import the modules**

In [1]:
#ifdef _WIN32
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteCore")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMarkovChain")
#pragma cling add_library_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/bin")
#pragma cling load("marmoteCore.dll")
#pragma cling load("marmoteMarkovChain.dll")
#else
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMarkovChain")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMarkovChain.so")
#endif

In [2]:
// --- Standard C++ utilities used in this notebook ---
#include <iostream>
#include <iomanip>
#include <string>
#include <vector>

// --- Marmote headers used in this lesson ---
#include <marmoteCore/marmoteCore>
#include <marmoteMarkovChain/marmoteMarkovChain>

// --- Convenience declarations for the cells below ---
using namespace std;
using namespace marmote;

State spaces for Marmote Markov chains are specified with `MarmoteSet` objects. This lesson presents the principal features of these objects and their main implementations.

## Functionalities of MarmoteSet: example with intervals and boxes

In [3]:
// Create an interval [0..6].
MarmoteInterval* itl = new MarmoteInterval(0, 6);
cout << "name = " << itl->className() << endl;
cout << "cardinal = " << itl->Cardinal() << endl;
cout << itl->toString() << endl;

// Create a 2-d box with sizes 2 and 3.
stateType dims[2] = {2, 3};
MarmoteBox* box = new MarmoteBox(2, dims);
cout << "name = " << box->className() << endl;
cout << "cardinal = " << box->Cardinal() << endl;
cout << box->toString() << endl;
cout << box->Enumerate() << endl;

name = MarmoteInterval
cardinal = 7
Interval( 0..6 )
name = MarmoteBox
cardinal = 6
Box( [ 0..1 ] x [ 0..2 ] )
(   0,   0) (   0,   1) (   0,   2) (   1,   0) (   1,   1) (   1,   2) 


## States and their indices

Every state has an index and conversely. In C++, `DecodeState` writes the decoded state into a buffer.

In [4]:
// Compute the index of state (1,0).
stateType st10[2] = {1, 0};
cout << box->Index(st10) << endl;

// Decode index 4 into a state buffer.
MarmoteState buf = box->StateBuffer();
box->DecodeState(4, buf);
cout << static_cast<MarmoteSet*>(box)->PrintState(buf, FORMAT_STRUCTURED) << endl;

// Change how states are printed.
stateWriteType polsave = marmote::stateWritePolicy();
marmote::setStateWritePolicy(STATE_INDEX);
cout << "Format with index: " << box->FormatState(4) << endl;
marmote::setStateWritePolicy(STATE_BOTH);
cout << "Format with both: " << box->FormatState(4) << endl;
marmote::setStateWritePolicy(STATE_FULL);
cout << "Format with full description: " << box->FormatState(4) << endl;
marmote::setStateWritePolicy(polsave);

3
(   1,   1)
Format with index: 4 
Format with both: 4:{   1   1} 
Format with full description:    1   1 


### Checking membership and looping through the states

In [5]:
// Membership tests on intervals and boxes.
stateType s4[1] = {4};
stateType s40[1] = {40};
stateType b44[2] = {4, 4};
stateType b11[2] = {1, 1};
cout << itl->Belongs(s4) << endl;
cout << itl->Belongs(s40) << endl;
cout << box->Belongs(b44) << endl;
cout << box->Belongs(b11) << endl;

// Enumerate the box through a state buffer.
marmote::setStateWritePolicy(STATE_BOTH);
MarmoteState bbuf = box->StateBuffer();
box->FirstState(bbuf);
bool isover = false;
while (!isover) {
    cout << box->FormatState(box->Index(bbuf)) << endl;
    box->NextState(bbuf);
    isover = box->IsFirst(bbuf);
}

1
0
0
1
0:{   0   0} 
1:{   0   1} 
2:{   0   2} 
3:{   1   0} 
4:{   1   1} 
5:{   1   2} 


## Other sets

The Python lesson also illustrates binary sequences, the set of all integers, and simplices.

In [6]:
// Binary sequences.
BinarySequence* biseq = new BinarySequence(6);
cout << "name = " << biseq->className() << endl;
cout << "cardinal = " << biseq->Cardinal() << endl;
cout << biseq->Enumerate() << endl;

MarmoteState sbuf = biseq->StateBuffer();
biseq->FirstState(sbuf);
isover = false;
while (!isover) {
    cout << biseq->FormatState(biseq->Index(sbuf)) << endl;
    biseq->NextState(sbuf);
    isover = biseq->IsFirst(sbuf);
}

stateType st1[6] = {1, 0, 1, 1, 0, 1};
stateType st2[6] = {2, 4, 1, 1, 0, 1};
cout << biseq->Belongs(st1) << " " << biseq->Index(st1) << endl;
cout << biseq->Belongs(st2) << " " << biseq->Index(st2) << endl;

name = BinarySequence
cardinal = 64
( 0 0 0 0 0 0) ( 0 0 0 0 0 1) ( 0 0 0 0 1 0) ( 0 0 0 0 1 1) ( 0 0 0 1 0 0) ( 0 0 0 1 0 1) ( 0 0 0 1 1 0) ( 0 0 0 1 1 1) ( 0 0 1 0 0 0) ( 0 0 1 0 0 1) ( 0 0 1 0 1 0) ( 0 0 1 0 1 1) ( 0 0 1 1 0 0) ( 0 0 1 1 0 1) ( 0 0 1 1 1 0) ( 0 0 1 1 1 1) ( 0 1 0 0 0 0) ( 0 1 0 0 0 1) ( 0 1 0 0 1 0) ( 0 1 0 0 1 1) ( 0 1 0 1 0 0) ( 0 1 0 1 0 1) ( 0 1 0 1 1 0) ( 0 1 0 1 1 1) ( 0 1 1 0 0 0) ( 0 1 1 0 0 1) ( 0 1 1 0 1 0) ( 0 1 1 0 1 1) ( 0 1 1 1 0 0) ( 0 1 1 1 0 1) ( 0 1 1 1 1 0) ( 0 1 1 1 1 1) ( 1 0 0 0 0 0) ( 1 0 0 0 0 1) ( 1 0 0 0 1 0) ( 1 0 0 0 1 1) ( 1 0 0 1 0 0) ( 1 0 0 1 0 1) ( 1 0 0 1 1 0) ( 1 0 0 1 1 1) ( 1 0 1 0 0 0) ( 1 0 1 0 0 1) ( 1 0 1 0 1 0) ( 1 0 1 0 1 1) ( 1 0 1 1 0 0) ( 1 0 1 1 0 1) ( 1 0 1 1 1 0) ( 1 0 1 1 1 1) ( 1 1 0 0 0 0) ( 1 1 0 0 0 1) ( 1 1 0 0 1 0) ( 1 1 0 0 1 1) ( 1 1 0 1 0 0) ( 1 1 0 1 0 1) ( 1 1 0 1 1 0) ( 1 1 0 1 1 1) ( 1 1 1 0 0 0) ( 1 1 1 0 0 1) ( 1 1 1 0 1 0) ( 1 1 1 0 1 1) ( 1 1 1 1 0 0) ( 1 1 1 1 0 1) ( 1 1 1 1 1 0) ( 1 1 1 1 1 1) 
0:{

In [7]:
// The set of all integers.
MarmoteIntegers* itg = new MarmoteIntegers();
cout << "name = " << itg->className() << endl;
cout << "cardinal = " << itg->Cardinal() << endl;
cout << itg->IsFinite() << endl;
stateType ip4[1] = {4};
stateType im3[1] = {-3};
cout << itg->Belongs(ip4) << endl;
cout << itg->Belongs(im3) << endl;
cout << itg->Index(im3) << endl;

// Simplices.
Simplex* splx = new Simplex(7, 3);
cout << splx->Enumerate() << endl;
stateType sx1[7] = {0, 0, 0, 0, 0, 0, 0};
stateType sx2[7] = {0, 0, 0, 0, 0, 0, 3};
stateType sx3[7] = {0, -1, 1, 0, 0, 0, 3};
cout << splx->Belongs(sx1) << endl;
cout << splx->Belongs(sx2) << endl;
cout << splx->Belongs(sx3) << endl;

BinarySimplex* bsplx = new BinarySimplex(7, 3);
cout << bsplx->Enumerate() << endl;

name = MarmoteIntegers
cardinal = -2
0
1
0
0
( 0 0 0 0 0 0 3) ( 0 0 0 0 0 1 2) ( 0 0 0 0 0 2 1) ( 0 0 0 0 0 3 0) ( 0 0 0 0 1 0 2) ( 0 0 0 0 1 1 1) ( 0 0 0 0 1 2 0) ( 0 0 0 0 2 0 1) ( 0 0 0 0 2 1 0) ( 0 0 0 0 3 0 0) ( 0 0 0 1 0 0 2) ( 0 0 0 1 0 1 1) ( 0 0 0 1 0 2 0) ( 0 0 0 1 1 0 1) ( 0 0 0 1 1 1 0) ( 0 0 0 1 2 0 0) ( 0 0 0 2 0 0 1) ( 0 0 0 2 0 1 0) ( 0 0 0 2 1 0 0) ( 0 0 0 3 0 0 0) ( 0 0 1 0 0 0 2) ( 0 0 1 0 0 1 1) ( 0 0 1 0 0 2 0) ( 0 0 1 0 1 0 1) ( 0 0 1 0 1 1 0) ( 0 0 1 0 2 0 0) ( 0 0 1 1 0 0 1) ( 0 0 1 1 0 1 0) ( 0 0 1 1 1 0 0) ( 0 0 1 2 0 0 0) ( 0 0 2 0 0 0 1) ( 0 0 2 0 0 1 0) ( 0 0 2 0 1 0 0) ( 0 0 2 1 0 0 0) ( 0 0 3 0 0 0 0) ( 0 1 0 0 0 0 2) ( 0 1 0 0 0 1 1) ( 0 1 0 0 0 2 0) ( 0 1 0 0 1 0 1) ( 0 1 0 0 1 1 0) ( 0 1 0 0 2 0 0) ( 0 1 0 1 0 0 1) ( 0 1 0 1 0 1 0) ( 0 1 0 1 1 0 0) ( 0 1 0 2 0 0 0) ( 0 1 1 0 0 0 1) ( 0 1 1 0 0 1 0) ( 0 1 1 0 1 0 0) ( 0 1 1 1 0 0 0) ( 0 1 2 0 0 0 0) ( 0 2 0 0 0 0 1) ( 0 2 0 0 0 1 0) ( 0 2 0 0 1 0 0) ( 0 2 0 1 0 0 0) ( 0 2 1 0 0 0 0) ( 0 3 0 0 0 0 0) ( 1

In [8]:
// Release the set objects and the state buffers allocated above.
delete bsplx;
delete splx;
delete itg;
delete[] sbuf;
delete biseq;
delete[] bbuf;
delete[] buf;
delete box;
delete itl;